# 🏋️ Notebook 4 — Predictive Modelling for Motion Outcome
**Input:** `final_dataset.csv`

Three models compared:
| # | Model | Feature set |
|---|---|---|
| 1 | **XGBoost** | 43 structured + 10 RobBERT features |
| 2 | **Logistic Regression** | Same features + StandardScaler |
| 3 | **RobBERT Sequence Classifier** | Raw Dutch topic title text |

**Target:** `label` — 1=Accepted, 0=Rejected  
**Metrics:** Accuracy · F1 · ROC-AUC · Brier score · Confusion matrix


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, brier_score_loss,
    confusion_matrix, classification_report, RocCurveDisplay,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import DataLoader, Dataset

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, RED, GREEN, ORANGE = '#2B5797','#C0392B','#27AE60','#E67E22'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

df = pd.read_csv("final_dataset.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")


In [ ]:
# ── Feature definitions ────────────────────────────────────────────────────
STRUCTURED_COLS = [
    'hour', 'tod_ordinal', 'hour_sin', 'hour_cos',
    'is_early_morning', 'is_morning', 'is_early_afternoon',
    'is_late_afternoon', 'is_evening', 'is_night',
    'years_since_2009', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'season_ordinal', 'is_monday', 'is_friday',
    'is_plenary', 'log_duration', 'log_density', 'high_density_day',
    'plenary_x_tod', 'plenary_x_night', 'high_density_x_night',
    'duration_x_density',
    'cat_governance_and_administration', 'cat_infrastructure_and_transport',
    'cat_economy_and_finance', 'cat_healthcare', 'cat_education',
    'cat_housing', 'cat_public_safety_and_policing',
    'cat_employment_and_labour', 'cat_environment_and_climate',
    'cat_social_welfare',
]

ROBBERT_COLS = [
    'sentiment_score', 'sentiment_label_enc',
    'tone_label_enc',
    'tone_aggressive', 'tone_mean', 'tone_neutral',
    'tone_peaceful', 'tone_kind', 'tone_happy',
]

ALL_FEAT_COLS = STRUCTURED_COLS + [c for c in ROBBERT_COLS if c in df.columns]
print(f"Structured features: {len(STRUCTURED_COLS)}")
print(f"RobBERT features available: {[c for c in ROBBERT_COLS if c in df.columns]}")
print(f"Total features: {len(ALL_FEAT_COLS)}")


In [ ]:
# ── Prepare splits ─────────────────────────────────────────────────────────
train_mask = df['split'] == 'train'
val_mask   = df['split'] == 'val'
test_mask  = df['split'] == 'test'

X = df[ALL_FEAT_COLS].fillna(0).astype(float)
y = df['label'].astype(int)

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Acceptance rate — Train: {y_train.mean():.3f} | Val: {y_val.mean():.3f} | Test: {y_test.mean():.3f}")


## Model 1 — XGBoost

In [ ]:
xgb_params = dict(
    n_estimators      = 500,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 5,
    gamma             = 0.1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    eval_metric       = 'logloss',
    random_state      = 42,
    n_jobs            = -1,
)

xgb = XGBClassifier(**xgb_params)
xgb.fit(X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100)
print("XGBoost training complete ✓")


In [ ]:
def evaluate_model(model, X, y, model_name, split_name):
    preds  = model.predict(X)
    probas = model.predict_proba(X)[:,1]
    return {
        'model':     model_name,
        'split':     split_name,
        'accuracy':  accuracy_score(y, preds),
        'f1':        f1_score(y, preds),
        'roc_auc':   roc_auc_score(y, probas),
        'brier':     brier_score_loss(y, probas),
        '_preds':    preds,
        '_probas':   probas,
        '_y':        y,
    }

xgb_val  = evaluate_model(xgb, X_val,  y_val,  'XGBoost', 'Val')
xgb_test = evaluate_model(xgb, X_test, y_test, 'XGBoost', 'Test')

print("\n=== XGBoost Test Results ===")
print(f"  Accuracy : {xgb_test['accuracy']:.4f}")
print(f"  F1       : {xgb_test['f1']:.4f}")
print(f"  ROC-AUC  : {xgb_test['roc_auc']:.4f}")
print(f"  Brier    : {xgb_test['brier']:.4f}")
print("\n", classification_report(y_test, xgb_test['_preds'],
                                   target_names=['Rejected','Accepted']))


In [ ]:
# ── XGBoost Feature Importance ────────────────────────────────────────────
fi = pd.Series(xgb.feature_importances_, index=ALL_FEAT_COLS).sort_values(ascending=True)
top_n = fi.tail(20)

fig, ax = plt.subplots(figsize=(8, 7))
colors = [GREEN if 'robbert' in i or 'sentiment' in i or 'tone' in i
          else BLUE for i in top_n.index]
ax.barh(top_n.index, top_n.values, color=colors, edgecolor='white', alpha=0.85)
ax.set_title('XGBoost — Top 20 Feature Importances\n(green = RobBERT-derived)',
             fontweight='bold')
ax.set_xlabel('Importance (gain)')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=BLUE,label='Structured'), 
                   Patch(facecolor=GREEN,label='RobBERT')], loc='lower right')
plt.tight_layout(); plt.show()


In [ ]:
# ── 5-fold CV ─────────────────────────────────────────────────────────────
X_tv = pd.concat([X_train, X_val])
y_tv = pd.concat([y_train, y_val])
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_aucs = []
for fold, (tr, va) in enumerate(cv.split(X_tv, y_tv)):
    m = XGBClassifier(**xgb_params)
    m.fit(X_tv.iloc[tr], y_tv.iloc[tr], verbose=False)
    p = m.predict_proba(X_tv.iloc[va])[:,1]
    auc = roc_auc_score(y_tv.iloc[va], p)
    cv_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC = {auc:.4f}")
print(f"\n  CV Mean AUC: {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")


## Model 2 — Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegressionCV(
        Cs=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        cv=5, scoring='roc_auc', penalty='l2',
        solver='lbfgs', max_iter=2000,
        random_state=42, n_jobs=-1,
    ))
])

X_tv = pd.concat([X_train, X_val])
y_tv = pd.concat([y_train, y_val])
lr_pipe.fit(X_tv, y_tv)

best_C = lr_pipe.named_steps['lr'].C_[0]
print(f"Logistic Regression trained ✓  |  Best C (CV): {best_C}")

lr_test = evaluate_model(lr_pipe, X_test, y_test, 'LogisticReg', 'Test')
print(f"\n=== Logistic Regression Test ===")
print(f"  Accuracy : {lr_test['accuracy']:.4f}")
print(f"  F1       : {lr_test['f1']:.4f}")
print(f"  ROC-AUC  : {lr_test['roc_auc']:.4f}")
print(classification_report(y_test, lr_test['_preds'],
                             target_names=['Rejected','Accepted']))


In [ ]:
# ── Logistic Regression coefficients ─────────────────────────────────────
coef = lr_pipe.named_steps['lr'].coef_[0]
coef_s = pd.Series(coef, index=ALL_FEAT_COLS).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, tail, title in [(axes[0], coef_s.head(15), 'Top 15 Negative (→ Rejection)'),
                         (axes[1], coef_s.tail(15), 'Top 15 Positive (→ Acceptance)')]:
    colors_c = [RED if v < 0 else GREEN for v in tail.values]
    ax.barh(tail.index, tail.values, color=colors_c, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.7)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Coefficient')
plt.tight_layout(); plt.show()


## Model 3 — RobBERT Sequence Classifier

Fine-tuning `pdelobelle/robbert-v2-dutch-base` directly on motion titles.

Architecture: RobBERT [CLS] → Dropout(0.1) → Linear(768 → 2)

Training strategy:
- Phase 1 (epochs 1–2): freeze RobBERT, train classifier head only
- Phase 2 (epochs 3–6): unfreeze all, fine-tune end-to-end with reduced LR


In [ ]:
ROBBERT_BASE = "pdelobelle/robbert-v2-dutch-base"

class MotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.enc = tokenizer(texts, truncation=True, padding='max_length',
                             max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

print(f"Loading RobBERT tokenizer: {ROBBERT_BASE}")
rb_tokenizer = AutoTokenizer.from_pretrained(ROBBERT_BASE)
print("Tokenizer loaded ✓")


In [ ]:
text_col = 'Topic_clean' if 'Topic_clean' in df.columns else 'Topic'
all_texts  = df[text_col].fillna('').tolist()
all_labels = df['label'].astype(int).tolist()
all_splits = df['split'].tolist()

def get_split(split_name):
    idx = [i for i, s in enumerate(all_splits) if s == split_name]
    return [all_texts[i] for i in idx], [all_labels[i] for i in idx]

tr_texts, tr_labels = get_split('train')
va_texts, va_labels = get_split('val')
te_texts, te_labels = get_split('test')

BATCH = 32
train_loader = DataLoader(MotionDataset(tr_texts, tr_labels, rb_tokenizer),
                           batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(MotionDataset(va_texts, va_labels, rb_tokenizer),
                           batch_size=BATCH)
test_loader  = DataLoader(MotionDataset(te_texts, te_labels, rb_tokenizer),
                           batch_size=BATCH)

print(f"DataLoaders ready — Train batches: {len(train_loader)}")


In [ ]:
print(f"Loading RobBERT model: {ROBBERT_BASE}")
rb_model = AutoModelForSequenceClassification.from_pretrained(ROBBERT_BASE, num_labels=2)
rb_model.to(DEVICE)
print(f"Model on {DEVICE} ✓  |  Parameters: {sum(p.numel() for p in rb_model.parameters()):,}")


In [ ]:
def eval_bert(model, loader):
    model.eval()
    preds_all, probs_all, labels_all = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbs  = batch['labels']
            out  = model(input_ids=ids, attention_mask=mask)
            pr   = torch.softmax(out.logits.cpu(), dim=-1)[:,1].numpy()
            pd_  = torch.argmax(out.logits.cpu(), dim=-1).numpy()
            preds_all.extend(pd_.tolist())
            probs_all.extend(pr.tolist())
            labels_all.extend(lbs.tolist())
    return (np.array(preds_all), np.array(probs_all), np.array(labels_all))

EPOCHS  = 6
LR      = 2e-5
WD      = 0.01
WARMUP  = 0.10

optimizer = torch.optim.AdamW(rb_model.parameters(), lr=LR, weight_decay=WD)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP), total_steps)

best_val_auc = 0.0
history = []

# ── Phase 1: freeze RobBERT, train head only ───────────────────────────────
print("Phase 1: Training classifier head (epochs 1–2) …")
for param in rb_model.roberta.parameters():
    param.requires_grad = False

for epoch in range(EPOCHS):
    if epoch == 2:
        print("\nPhase 2: Unfreezing RobBERT for full fine-tuning …")
        for param in rb_model.roberta.parameters():
            param.requires_grad = True
        for g in optimizer.param_groups:
            g['lr'] = LR / 5     # lower LR for transformer layers

    rb_model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbs  = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out  = rb_model(input_ids=ids, attention_mask=mask, labels=lbs)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(rb_model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()

    avg_loss = total_loss / len(train_loader)
    vp, vpr, vy = eval_bert(rb_model, val_loader)
    val_auc = roc_auc_score(vy, vpr)
    val_f1  = f1_score(vy, vp)
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_auc': val_auc, 'val_f1': val_f1})
    print(f"  Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | val_auc={val_auc:.4f} | val_f1={val_f1:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(rb_model.state_dict(), "robbert_best.pt")
        print(f"    → Best model saved (val_auc={best_val_auc:.4f})")

print("\nRobBERT training complete ✓")


In [ ]:
# ── Evaluate best checkpoint ──────────────────────────────────────────────
rb_model.load_state_dict(torch.load("robbert_best.pt", map_location=DEVICE))
rb_preds, rb_probs, rb_labels = eval_bert(rb_model, test_loader)

rb_test = {
    'model':    'RobBERT',
    'split':    'Test',
    'accuracy': accuracy_score(rb_labels, rb_preds),
    'f1':       f1_score(rb_labels, rb_preds),
    'roc_auc':  roc_auc_score(rb_labels, rb_probs),
    'brier':    brier_score_loss(rb_labels, rb_probs),
    '_preds':   rb_preds,
    '_probas':  rb_probs,
    '_y':       rb_labels,
}

print("=== RobBERT Test Results ===")
print(f"  Accuracy : {rb_test['accuracy']:.4f}")
print(f"  F1       : {rb_test['f1']:.4f}")
print(f"  ROC-AUC  : {rb_test['roc_auc']:.4f}")
print(f"  Brier    : {rb_test['brier']:.4f}")
print(classification_report(rb_labels, rb_preds, target_names=['Rejected','Accepted']))


In [ ]:
# ── Training curve ────────────────────────────────────────────────────────
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['loss'], marker='o', color=BLUE, lw=2)
axes[0].set_title('RobBERT Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].plot(hist_df['epoch'], hist_df['val_auc'], marker='o', color=GREEN, lw=2, label='Val AUC')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'], marker='s', color=ORANGE, lw=2, label='Val F1')
axes[1].axvline(2.5, color='grey', linestyle=':', label='Unfreeze point')
axes[1].set_title('RobBERT Validation Metrics', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()


## Model Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
results_list = [xgb_test, lr_test, rb_test]
summary = pd.DataFrame([{
    'Model':    r['model'],
    'Accuracy': r['accuracy'],
    'F1':       r['f1'],
    'ROC-AUC':  r['roc_auc'],
    'Brier ↓':  r['brier'],
} for r in results_list]).sort_values('ROC-AUC', ascending=False)

print("\n" + "="*58)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("="*58)


In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
ax = axes[0]
for r, color, ls in [(xgb_test, BLUE, '-'), (lr_test, ORANGE, '--'), (rb_test, GREEN, ':')]:
    RocCurveDisplay.from_predictions(
        r['_y'], r['_probas'], name=f"{r['model']} (AUC={r['roc_auc']:.3f})",
        ax=ax, color=color, linestyle=ls)
ax.set_title('ROC Curves — Test Set', fontweight='bold')
ax.plot([0,1],[0,1],'k--',alpha=0.3)

# Bar comparison
ax2 = axes[1]
metrics = ['Accuracy','F1','ROC-AUC']
x = np.arange(len(metrics))
w = 0.22
for i, (r, color) in enumerate([(xgb_test, BLUE),(lr_test, ORANGE),(rb_test, GREEN)]):
    vals = [r['accuracy'], r['f1'], r['roc_auc']]
    bars = ax2.bar(x + i*w, vals, w, label=r['model'], color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f"{bar.get_height():.3f}", ha='center', fontsize=7)
ax2.set_xticks(x + w); ax2.set_xticklabels(metrics)
ax2.set_ylim(0, 1.05); ax2.set_title('Metric Comparison — Test Set', fontweight='bold')
ax2.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, r in zip(axes, [xgb_test, lr_test, rb_test]):
    ConfusionMatrixDisplay.from_predictions(
        r['_y'], r['_preds'],
        display_labels=['Rejected','Accepted'],
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['model']}", fontweight='bold')
plt.suptitle('Confusion Matrices — Test Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── Save results JSON ─────────────────────────────────────────────────────
import os
os.makedirs("outputs", exist_ok=True)

for r in results_list:
    out = {
        'model':      r['model'],
        'accuracy':   r['accuracy'],
        'f1':         r['f1'],
        'roc_auc':    r['roc_auc'],
        'brier':      r['brier'],
    }
    fname = f"outputs/{r['model'].lower().replace(' ','_')}_results.json"
    with open(fname, 'w') as f:
        json.dump(out, f, indent=2)
    print(f"Saved {fname}")
